In [ ]:
# Colab/local bootstrap: mount Drive (if needed), clone/install package, init wandb.
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/haydenyoungcs/gradient-ascent.git"
REPO_DIR = pathlib.Path("/content/gradient-ascent")
DEFAULT_COLAB_OUT_DIR = pathlib.Path("/content/drive/MyDrive/gradient-ascent-out")

github_token = os.environ.get("GITHUB_TOKEN")
wandb_api_key = os.environ.get("WANDB_API_KEY")
IN_COLAB = False

try:
    from google.colab import drive, userdata  # type: ignore

    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)

    if github_token is None:
        github_token = userdata.get("GITHUB_TOKEN")
    if wandb_api_key is None:
        wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    pass


def _find_project_root_from_cwd() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return None


project_root = _find_project_root_from_cwd() or REPO_DIR
if not project_root.exists():
    if github_token:
        clone_url = REPO_URL.replace("https://", f"https://{github_token}@")
        subprocess.run(["git", "clone", clone_url], check=True)
    else:
        raise RuntimeError(
            "Repo checkout not found. Add GITHUB_TOKEN as env var/Colab secret, or clone manually."
        )

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected pyproject.toml under {project_root}, but it was not found.")

repo_src = project_root / "src"
for path in [project_root, repo_src]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gradient_ascent.notebook_bootstrap import bootstrap_notebook_environment

bootstrap = bootstrap_notebook_environment(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    default_colab_out_dir=DEFAULT_COLAB_OUT_DIR,
    github_token=github_token,
    wandb_api_key=wandb_api_key,
)

project_root = bootstrap.project_root
IN_COLAB = bootstrap.in_colab
DEFAULT_OUT_DIR = bootstrap.default_out_dir
wandb = bootstrap.wandb

print(f"Changed working directory to {project_root}")
print(f"Default OUT_DIR: {DEFAULT_OUT_DIR}")


## Section 1 - Core training and unlearning checkpoints

This section trains the original and retrained models, then runs five unlearning baselines: gradient ascent, SSD, SalUn, certified removal, and SCRUB. For GA, the preset below stays faithful to vanilla gradient ascent on the forget set, but is made slightly stronger in an easy-to-justify way: a modestly higher learning rate, more forget-set updates per epoch, BatchNorm buffers still frozen for clean evaluation, and a looser clip so updates are visible without becoming unstable.

It also saves simple baseline-specific diagnostics that are easy to explain in a dissertation. For GA, these are the mean forget-set cross-entropy and gradient norm at each unlearning step. For SCRUB, they are the student-teacher KL divergence on the forget and retain sets, the retain-set cross-entropy, and retain/forget accuracy. Together these show whether SCRUB is separating from the teacher on forgotten data while still staying close on retained data.

In [ ]:
import warnings

from gradient_ascent.notebook_helpers import (
    prepare_notebook_runtime,
    run_and_display_notebook_core_pipeline,
)

warnings.filterwarnings("ignore", category=DeprecationWarning)

NUM_CLASSES = 10
OUT_DIR = DEFAULT_OUT_DIR
REUSE_EXISTING_CHECKPOINTS = True
REUSE_ORIGINAL_CHECKPOINT = True
REUSE_RETRAINED_CHECKPOINT = True  # Set False to force retraining the without-frog model only.
REUSE_UNLEARNED_CHECKPOINTS = True  # Reuse saved GA/SSD/SalUn/Certified/SCRUB outputs when available.
RUN_CORE_DIAGNOSTICS = False  # Set True when you want GA/SCRUB diagnostic CSV+plots.
resnet_model_depth = 50

runtime = prepare_notebook_runtime(
    num_classes=NUM_CLASSES,
    out_dir=OUT_DIR,
    model_depth=resnet_model_depth,
)

device = runtime.device
USE_BF16 = runtime.use_bf16
trainset = runtime.trainset
testset = runtime.testset
use_cuda = runtime.use_cuda
num_workers = runtime.num_workers
model_factory = runtime.model_factory
core_config = runtime.core_config

print(f"Runtime prepared on device: {device}")

core_artifacts, wandb_run = run_and_display_notebook_core_pipeline(
    runtime,
    wandb_module=wandb,
    reuse_existing_checkpoints=REUSE_EXISTING_CHECKPOINTS,
    reuse_original_checkpoint=REUSE_ORIGINAL_CHECKPOINT,
    reuse_retrained_checkpoint=REUSE_RETRAINED_CHECKPOINT,
    reuse_unlearned_checkpoints=REUSE_UNLEARNED_CHECKPOINTS,
    run_diagnostics=RUN_CORE_DIAGNOSTICS,
)


## Section 2 - Trajectories, MIA, and evolving similarity bar plots

Initialises shared similarity helpers, then runs snapshot trajectories for all five baselines (GA, SSD, SalUn, certified removal, and SCRUB), computes MIA trajectories, and generates PDF-friendly similarity summaries plus two evolving bar visualisations for both references: (1) mean-across-layers bars and (2) grouped bars by metric with per-layer bars.

In [ ]:
# Unlearning algorithm comparison trajectories across all five baselines.
# Shared similarity setup is created here so this cell is self-contained.

from gradient_ascent.notebook_helpers import (
    prepare_similarity_setup,
    run_and_display_notebook_trajectory_pipeline,
)

similarity_setup = prepare_similarity_setup()
layer_names = similarity_setup.layer_names
metrics = similarity_setup.metrics
higher_better_metrics = similarity_setup.higher_better_metrics
lower_better_metrics = similarity_setup.lower_better_metrics
plot_metric_names = similarity_setup.plot_metric_names

trajectory_artifacts, trajectory_wandb_run = run_and_display_notebook_trajectory_pipeline(
    runtime,
    core_artifacts,
    similarity_setup,
    wandb_module=wandb,
)


## Section 3 - Combined cross-algorithm comparison

Overlays all five baselines, including SCRUB, in a single consolidated similarity + MIA comparison figure.

In [ ]:
# Integrated combined comparison figure: similarity + MIA trajectories.

from gradient_ascent.notebook_helpers import run_and_display_notebook_combined_comparison

combined_path = run_and_display_notebook_combined_comparison(runtime, wandb_module=wandb)


## Section 4 - Multi-target averaging across all forget classes

Runs the full pipeline for each forget label (`0..9`) and then aggregates outputs:

- Utility plots become two-series curves: `forgotten class` and `retained classes (mean)`.
- Similarity trajectories are averaged over forget labels for each algorithm/reference pair.

This section is compute-heavy because it effectively multiplies core + trajectory work by 10.

In [ ]:
# Optional one-time migration: copy existing single-target frog outputs into
# multi-target target_6 folder so reruns can reuse them.

from pathlib import Path
import shutil

single_out = Path(OUT_DIR)
multi_root = Path(f"{OUT_DIR}/multitarget_aggregate")
shared_dir = multi_root / "shared"
shared_dir.mkdir(parents=True, exist_ok=True)
shared_original = shared_dir / "original_net.pt"

# Ensure the canonical shared original checkpoint exists for multi-target runs.
src_original = single_out / "original_net.pt"
if src_original.exists() and not shared_original.exists():
    shutil.copy2(src_original, shared_original)
    print(f"Copied shared original checkpoint to: {shared_original}")

if not shared_original.exists():
    raise FileNotFoundError(
        f"Missing shared original checkpoint at {shared_original}. "
        "Place/copy your pretrained original model there before running Section 4."
    )

target_label = 6  # frog
target_out = multi_root / f"target_{target_label}"
target_out.mkdir(parents=True, exist_ok=True)

file_mappings = {
    "original_net.pt": "original_net.pt",
    "retrained_from_scratch_net.pt": "retrained_from_scratch_net.pt",
    "original_vs_retrain_acc.png": "original_vs_retrain_acc.png",
    "classwise_accuracy_original.png": "classwise_accuracy_original.png",
    "classwise_accuracy_retrained.png": "classwise_accuracy_retrained.png",
    "classwise_percent_diff_retrained_vs_original.png": "classwise_percent_diff_retrained_vs_original.png",
    "unlearning_runtime_seconds.csv": "unlearning_runtime_seconds.csv",
    "unlearning_runtime_seconds.png": "unlearning_runtime_seconds.png",
    "mia_retrained_baseline.csv": "mia_retrained_baseline.csv",
    "trajectory_timing_seconds.csv": "trajectory_timing_seconds.csv",
}

for algo in ["ga", "ssd", "salun", "certified", "scrub"]:
    file_mappings[f"unlearned_net_{algo}.pt"] = f"unlearned_net_{algo}.pt"
    file_mappings[f"classwise_accuracy_{algo}.csv"] = f"classwise_accuracy_{algo}.csv"
    file_mappings[f"classwise_percent_change_{algo}.png"] = f"classwise_percent_change_{algo}.png"
    file_mappings[f"classwise_absolute_accuracy_{algo}.png"] = f"classwise_absolute_accuracy_{algo}.png"
    file_mappings[f"mia_vs_unlearning_epoch_{algo}.csv"] = f"mia_vs_unlearning_epoch_{algo}.csv"
    file_mappings[f"mia_frog_trajectory_{algo}.png"] = f"mia_frog_trajectory_{algo}.png"
    file_mappings[f"mia_forget_vs_retain_logreg_mean_member_prob_{algo}.png"] = (
        f"mia_forget_vs_retain_logreg_mean_member_prob_{algo}.png"
    )
    file_mappings[f"similarity_metric_timing_{algo}.png"] = f"similarity_metric_timing_{algo}.png"
    for ref in ["retrained", "original"]:
        prefix = f"similarity_vs_unlearning_epoch_{algo}_vs_{ref}"
        file_mappings[f"{prefix}.csv"] = f"{prefix}.csv"
        file_mappings[f"{prefix}_summary.png"] = f"{prefix}_summary.png"
        file_mappings[f"{prefix}_evolving_bars.gif"] = f"{prefix}_evolving_bars.gif"
        file_mappings[f"{prefix}_evolving_grouped_bars.gif"] = f"{prefix}_evolving_grouped_bars.gif"
        file_mappings[f"{prefix}_before_after_grouped_bars.png"] = f"{prefix}_before_after_grouped_bars.png"

# Snapshot directories and optional activation cache directories.
dir_mappings = {
    "unlearning_snapshots_ga": "unlearning_snapshots_ga",
    "unlearning_snapshots_ssd": "unlearning_snapshots_ssd",
    "unlearning_snapshots_salun": "unlearning_snapshots_salun",
    "unlearning_snapshots_certified": "unlearning_snapshots_certified",
    "unlearning_snapshots_scrub": "unlearning_snapshots_scrub",
    "similarity_activation_cache": "similarity_activation_cache",
}

copied_files = 0
skipped_files = 0
for src_name, dst_name in file_mappings.items():
    src = single_out / src_name
    dst = target_out / dst_name
    if not src.exists():
        continue
    if dst.exists():
        skipped_files += 1
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied_files += 1

copied_dirs = 0
skipped_dirs = 0
for src_name, dst_name in dir_mappings.items():
    src = single_out / src_name
    dst = target_out / dst_name
    if not src.exists() or not src.is_dir():
        continue
    if dst.exists():
        skipped_dirs += 1
        continue
    shutil.copytree(src, dst)
    copied_dirs += 1

print(f"Frog migration complete: copied_files={copied_files}, skipped_files={skipped_files}, copied_dirs={copied_dirs}, skipped_dirs={skipped_dirs}")
print(f"Target folder: {target_out}")

In [ ]:
from gradient_ascent.notebook_helpers import (
    prepare_similarity_setup,
    run_multitarget_averaged_experiment,
)

# Shared similarity setup so this cell is self-contained (same as Section 2).
similarity_setup = prepare_similarity_setup()

# Set this to False if you already ran per-target trajectories and only want
# to recompute aggregate plots/CSVs from existing outputs.
RUN_SIMILARITY_STAGE = True

multi_target_artifacts = run_multitarget_averaged_experiment(
    runtime,
    similarity_setup,
    target_labels=list(range(10)),
    out_dir=f"{OUT_DIR}/multitarget_aggregate",
    run_similarity_stage=RUN_SIMILARITY_STAGE,
    wandb_module=wandb,
    reuse_existing_checkpoints=REUSE_EXISTING_CHECKPOINTS,
    reuse_original_checkpoint=REUSE_ORIGINAL_CHECKPOINT,
    reuse_retrained_checkpoint=REUSE_RETRAINED_CHECKPOINT,
    reuse_unlearned_checkpoints=REUSE_UNLEARNED_CHECKPOINTS,
    reuse_trajectory_outputs=True,
    shared_original_checkpoint_path=f"{OUT_DIR}/multitarget_aggregate/shared/original_net.pt",
)

print("Saved multi-target aggregate artefacts under:", f"{OUT_DIR}/multitarget_aggregate")
print("Utility aggregate CSVs:")
for algorithm_key, csv_path in multi_target_artifacts.utility_csv_paths.items():
    print(f"  {algorithm_key}: {csv_path}")

if RUN_SIMILARITY_STAGE:
    print("Averaged MIA baseline CSV:", multi_target_artifacts.mia_baseline_csv_path)
    print("Averaged MIA trajectory CSVs:")
    for algorithm_key, csv_path in multi_target_artifacts.mia_csv_paths.items():
        print(f"  {algorithm_key}: {csv_path}")